# Задание

**Изучить и реализовать MLP (многослойная искусственная нейронная сеть) для классификации нелинейно разделимых данных.**

1. Сгенерируйте данные (любым методом из sklearn.datasets).
2. Визуализируйте данные на плоскости.
3. Создайте MLP: вход (2 нейрона), скрытый слой (4 нейрона, ReLU), выход (1 нейрон, sigmoid) - можете использовать pytorch / keras
4. Разделите данные (80% — обучение, 20% — тест).
5. Обучите на 1000 эпох, постройте график потерь.
6. Оцените точность на тесте.
7. Если увлеклись, можно реализовать MLP для решения XOR.

# Импорт библиотек

Загружаем необходимые библиотеки для обработки данных, визуализации и машинного обучения.

In [58]:
%pip install pandas numpy plotly nbformat scikit-learn torch


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [59]:
from sklearn.datasets import make_classification
import torch
import plotly.graph_objects as go
import numpy as np
from sklearn.model_selection import train_test_split

# Константы

In [60]:
RANDOM_SEED=12

SAMPLES_COUNT=1000
FEATURES_COUNT=2
CLASSES_COUNT=2

FIRST_LAYER_SIZE=2
SECOND_LAYER_SIZE=4

OUTPUT_LAYER_SIZE=1

EPOCHS=1000
LEARNING_RATE=1e-2

TEST_SPLIT=0.2

# Генерируем ДатаСет

In [61]:
X , y = make_classification(
    n_samples=SAMPLES_COUNT, 
    n_features=FEATURES_COUNT, 
    n_classes=CLASSES_COUNT,
    n_informative=2, 
    n_redundant=0,
    n_repeated=0,

    random_state=RANDOM_SEED
)

x_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=X[:, 0], 
    y=X[:, 1], 
    mode='markers', 
    marker=dict(color=y, colorscale='Viridis',
)))
fig.update_layout(title='Generated Data', xaxis_title='Feature 1', yaxis_title='Feature 2')
fig.show()

# MLP

In [62]:


class Linear:
    def __init__(self, in_features, out_features):
        self.W = np.random.randn(in_features, out_features) * 0.01
        self.b = np.zeros((1, out_features))
        self.dW = None
        self.db = None
        self.x = None
    
    def forward(self, x):
        self.x = x
        return np.dot(x, self.W) + self.b
    
    def backward(self, dout):
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0, keepdims=True)
        return np.dot(dout, self.W.T)
    
    def update(self, lr):
        self.W -= lr * self.dW
        self.b -= lr * self.db

class ReLU:
    def __init__(self):
        self.x = None
    
    def forward(self, x):
        self.x = x
        return np.maximum(0, x)
    
    def backward(self, dout):
        return dout * (self.x > 0)

class Sigmoid:
    def __init__(self):
        self.out = None
    
    def forward(self, x):
        self.out = 1 / (1 + np.exp(-x))
        return self.out
    
    def backward(self, dout):
        return dout * self.out * (1 - self.out)

class BCELoss:
    def forward(self, y_pred, y_true):
        eps = 1e-7
        return -np.mean(
            y_true * np.log(y_pred + eps)
            + (1 - y_true) * np.log(1 - y_pred + eps)
        )
    
    def backward(self, y_pred, y_true):
        eps = 1e-7
        return (-(y_true / (y_pred + eps))
                + (1 - y_true) / (1 - y_pred + eps)) / len(y_true)
class MLP:
    def __init__(self, input_size, hidden_size, output_size):
        self.fc1 = Linear(input_size, hidden_size)
        self.relu = ReLU()
        self.fc2 = Linear(hidden_size, output_size)
        self.sigmoid = Sigmoid()
        self.loss_fn = BCELoss()
    
    def forward(self, x):
        h = self.fc1.forward(x)
        h = self.relu.forward(h)
        out = self.fc2.forward(h)
        out = self.sigmoid.forward(out)
        return out
    
    def backward(self, dout):
        dout = self.sigmoid.backward(dout)
        dout = self.fc2.backward(dout)
        dout = self.relu.backward(dout)
        dout = self.fc1.backward(dout)
    
    def update(self, lr):
        self.fc1.update(lr)
        self.fc2.update(lr)
    
    def train_step(self, x, y, lr):
        out = self.forward(x)
        loss = self.loss_fn.forward(out, y)
        dloss = self.loss_fn.backward(out, y)
        self.backward(dloss)
        self.update(lr)
        return loss
    
    def predict(self, x):
        return self.forward(x) > 0.5


In [63]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SPLIT, random_state=RANDOM_SEED
)

In [64]:
losses = []
mlp = MLP(FEATURES_COUNT, SECOND_LAYER_SIZE, OUTPUT_LAYER_SIZE)

for _ in range(EPOCHS):
    
    loss = mlp.train_step(X_train, y_train.reshape(-1, 1), LEARNING_RATE)
    losses.append(loss)


fig = go.Figure()
fig.add_trace(go.Scatter(
    y=losses, 
    mode='lines', 
    name='Training Loss'
))
fig.update_layout(title='Training Loss over Epochs', xaxis_title='Epoch', yaxis_title='Loss')
fig.show()

In [65]:
accuracy = np.mean(mlp.predict(X_test).flatten() == y_test)
print(f'Accuracy: {accuracy * 100:.2f}%')

Accuracy: 52.00%
